# 05 — Lakeview Dashboard (AI/BI)

Creates a published Lakeview dashboard backed by the gold star schema —
parameterized by `SQL_WAREHOUSE_ID`, idempotent on re-run.

**5 datasets** × **6 widgets** on a single page:

| Row | Widgets |
|-----|---------|
| Top  (0–3) | 3 KPI counters: Artists · Companies · People |
| Mid  (3–9) | Agency market share (pie) · Top managers (table) |
| Bot (9–15) | Artists by Spotify followers (bar) · Top labels (bar) |

In [1]:
import os, json
from dotenv import load_dotenv
load_dotenv()

try:
    spark
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.dashboards import Dashboard
w = WorkspaceClient()

In [2]:
UC_CATALOG  = os.getenv('UC_CATALOG', 'alexander_booth')
UC_SCHEMA   = os.getenv('UC_SCHEMA',  'rostr_music_industry')
GOLD_SCHEMA = f'{UC_SCHEMA}_gold'
G = f'{UC_CATALOG}.{GOLD_SCHEMA}'

SQL_WAREHOUSE_ID = os.getenv('SQL_WAREHOUSE_ID')
if not SQL_WAREHOUSE_ID:
    raise ValueError('SQL_WAREHOUSE_ID not set in .env')

DASHBOARD_NAME = 'rostr.cc — Music Industry Map'
print(f'Gold:      {G}')
print(f'Warehouse: {SQL_WAREHOUSE_ID}')

Gold:      alexander_booth.rostr_music_industry_gold
Warehouse: e9b34f7a2e4b0561


In [3]:
datasets = [
    {'name': 'kpi_summary', 'displayName': 'KPIs',
     'queryLines': [
         'SELECT\n',
         '  (SELECT COUNT(*) FROM ' + G + '.dim_artist)  AS artists,\n',
         '  (SELECT COUNT(*) FROM ' + G + '.dim_company) AS companies,\n',
         '  (SELECT COUNT(*) FROM ' + G + '.dim_person)  AS people',
     ]},
    {'name': 'agency_share', 'displayName': 'Agency Market Share',
     'queryLines': [
         'SELECT agency, artists_represented, market_share_pct\n',
         f'FROM {G}.v_agency_market_share',
     ]},
    {'name': 'top_managers', 'displayName': 'Top Managers',
     'queryLines': [
         'SELECT person_name, company_name, artist_count\n',
         f'FROM {G}.v_top_managers\n',
         'LIMIT 25',
     ]},
    {'name': 'spotify_followers', 'displayName': 'Artists by Spotify Followers',
     'queryLines': [
         'SELECT artist_name, spotify_followers\n',
         f'FROM {G}.dim_artist\n',
         'WHERE spotify_followers IS NOT NULL\n',
         'ORDER BY spotify_followers DESC',
     ]},
    {'name': 'top_labels', 'displayName': 'Top Labels',
     'queryLines': [
         'SELECT label, label_type, artist_count\n',
         f'FROM {G}.v_artists_by_label',
     ]},
]

In [4]:
def counter(name, dataset, field, title, x, y, w_=4, h=3):
    return {
        'widget': {
            'name': name,
            'queries': [{'name': 'main_query', 'query': {
                'datasetName': dataset,
                'fields': [{'name': field, 'expression': f'`{field}`'}],
                'disaggregated': True,
            }}],
            'spec': {
                'version': 2, 'widgetType': 'counter',
                'encodings': {'value': {'fieldName': field, 'displayName': title}},
                'frame': {'showTitle': True, 'title': title},
            },
        },
        'position': {'x': x, 'y': y, 'width': w_, 'height': h},
    }

def chart(name, dataset, widget_type, encodings, fields, x, y, w_, h, *, title=None, disaggregated=False):
    return {
        'widget': {
            'name': name,
            'queries': [{'name': 'main_query', 'query': {
                'datasetName': dataset, 'fields': fields, 'disaggregated': disaggregated,
            }}],
            'spec': {
                'version': 3, 'widgetType': widget_type,
                'encodings': encodings,
                'frame': {'showTitle': True, 'title': title or name},
            },
        },
        'position': {'x': x, 'y': y, 'width': w_, 'height': h},
    }

def table(name, dataset, columns, x, y, w_, h, *, title=None):
    fields = [{'name': c['fieldName'], 'expression': f'`{c["fieldName"]}`'} for c in columns]
    return {
        'widget': {
            'name': name,
            'queries': [{'name': 'main_query', 'query': {
                'datasetName': dataset, 'fields': fields, 'disaggregated': True,
            }}],
            'spec': {
                'version': 1, 'widgetType': 'table',
                'encodings': {'columns': columns},
                'frame': {'showTitle': True, 'title': title or name},
            },
        },
        'position': {'x': x, 'y': y, 'width': w_, 'height': h},
    }

In [5]:
layout = [
    counter('kpi_artists',   'kpi_summary', 'artists',   'Artists',   0, 0),
    counter('kpi_companies', 'kpi_summary', 'companies', 'Companies', 4, 0),
    counter('kpi_people',    'kpi_summary', 'people',    'People',    8, 0),

    chart(
        'agency_pie', 'agency_share', 'pie',
        encodings={
            'angle': {'fieldName': 'artists_represented', 'displayName': 'Artists'},
            'color': {'fieldName': 'agency', 'displayName': 'Agency',
                      'scale': {'type': 'categorical'}},
        },
        fields=[
            {'name': 'agency',                'expression': '`agency`'},
            {'name': 'artists_represented',   'expression': '`artists_represented`'},
        ],
        x=0, y=3, w_=6, h=6, title='Agency Market Share', disaggregated=True,
    ),
    table(
        'managers_table', 'top_managers',
        columns=[
            {'fieldName': 'person_name',  'displayName': 'Manager',     'type': 'string'},
            {'fieldName': 'company_name', 'displayName': 'Firm',        'type': 'string'},
            {'fieldName': 'artist_count', 'displayName': 'Artists',     'type': 'integer'},
        ],
        x=6, y=3, w_=6, h=6, title='Top Managers',
    ),

    chart(
        'followers_bar', 'spotify_followers', 'bar',
        encodings={
            'x': {'fieldName': 'artist_name', 'displayName': 'Artist',
                  'scale': {'type': 'categorical', 'sort': {'by': 'y-reversed'}}},
            'y': {'fieldName': 'spotify_followers', 'displayName': 'Spotify Followers',
                  'scale': {'type': 'quantitative'}},
        },
        fields=[
            {'name': 'artist_name',        'expression': '`artist_name`'},
            {'name': 'spotify_followers',  'expression': '`spotify_followers`'},
        ],
        x=0, y=9, w_=6, h=6, title='Artists by Spotify Followers', disaggregated=True,
    ),
    chart(
        'labels_bar', 'top_labels', 'bar',
        encodings={
            'x': {'fieldName': 'label', 'displayName': 'Label',
                  'scale': {'type': 'categorical', 'sort': {'by': 'y-reversed'}}},
            'y': {'fieldName': 'artist_count', 'displayName': 'Artists',
                  'scale': {'type': 'quantitative'}},
            'color': {'fieldName': 'label_type', 'displayName': 'Type',
                      'scale': {'type': 'categorical'}},
        },
        fields=[
            {'name': 'label',         'expression': '`label`'},
            {'name': 'artist_count',  'expression': '`artist_count`'},
            {'name': 'label_type',    'expression': '`label_type`'},
        ],
        x=6, y=9, w_=6, h=6, title='Top Labels by Roster Size', disaggregated=True,
    ),
]

dashboard_json = {
    'datasets': datasets,
    'pages': [{'name': 'overview', 'displayName': 'rostr Overview', 'layout': layout}],
}
serialized = json.dumps(dashboard_json)
print(f'datasets: {len(datasets)} | widgets: {len(layout)} | serialized: {len(serialized):,} chars')

datasets: 5 | widgets: 7 | serialized: 5,460 chars


In [6]:
current_user = w.current_user.me()
parent_path  = f'/Workspace/Users/{current_user.user_name}'

existing_id = None
try:
    for obj in w.workspace.list(parent_path):
        if obj.path and obj.path.endswith(f'{DASHBOARD_NAME}.lvdash.json'):
            existing_id = str(obj.object_id)
            break
except Exception:
    pass

if existing_id:
    try:
        d = w.lakeview.get(dashboard_id=existing_id)
        w.lakeview.update(
            dashboard_id=d.dashboard_id,
            dashboard=Dashboard(serialized_dashboard=serialized, warehouse_id=SQL_WAREHOUSE_ID),
        )
        dashboard_id = d.dashboard_id
        print(f'Dashboard updated in place: {dashboard_id}')
    except Exception:
        existing_id = None

if not existing_id:
    d = w.lakeview.create(
        dashboard=Dashboard(
            display_name=DASHBOARD_NAME,
            warehouse_id=SQL_WAREHOUSE_ID,
            serialized_dashboard=serialized,
            parent_path=parent_path,
        )
    )
    dashboard_id = d.dashboard_id
    print(f'Dashboard created: {dashboard_id}')

w.lakeview.publish(dashboard_id=dashboard_id, warehouse_id=SQL_WAREHOUSE_ID, embed_credentials=True)
host = os.getenv('DATABRICKS_HOST', '').rstrip('/')
print(f'\nView at: {host}/sql/dashboardsv3/{dashboard_id}')

Dashboard updated in place: 01f14aef19991611b591ac3ec8a8804c



View at: https://e2-demo-field-eng.cloud.databricks.com/sql/dashboardsv3/01f14aef19991611b591ac3ec8a8804c
